# MCP Text2SQL Project Tests

This notebook runs practical checks across guard, logger, DB, tools, API routes, and optional OpenAI call.

In [1]:
import json
import os
import uuid
from pathlib import Path
import sys

# Ensure project root is importable when running from test/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "test" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
from fastapi.testclient import TestClient
from sqlalchemy import text

from src import logger
from src import db_connection
from src import guard
from src import tools
import mcp_server

# Make sure .env is loaded for notebook tests
load_dotenv()

RESULTS = []

def add_result(name: str, status: str, details: str = ""):
    RESULTS.append({"test": name, "status": status, "details": details})

def mark_ok(name: str, details: str = ""):
    add_result(name, "PASS", details)

def mark_fail(name: str, details: str = ""):
    add_result(name, "FAIL", details)

def mark_skip(name: str, details: str = ""):
    add_result(name, "SKIP", details)

print("Imports and test harness ready")

Imports and test harness ready


In [2]:
# guard.py tests
try:
    out = guard.sanitize_and_validate_sql("SELECT name FROM sys.tables")
    if "TOP" in out.upper():
        mark_ok("guard_adds_top", out)
    else:
        mark_fail("guard_adds_top", f"Unexpected output: {out}")
except Exception as e:
    mark_fail("guard_adds_top", repr(e))

try:
    bad_q = "SELECT * FROM a JOIN b ON a.id=b.id JOIN c ON b.id=c.id JOIN d ON c.id=d.id"
    guard.sanitize_and_validate_sql(bad_q)
    mark_fail("guard_limits_complexity", "Expected ValueError but query passed")
except ValueError as e:
    if "max table limit" in str(e).lower() or "max complexity" in str(e).lower():
        mark_ok("guard_limits_complexity", str(e))
    else:
        mark_fail("guard_limits_complexity", f"Wrong error: {e}")
except Exception as e:
    mark_fail("guard_limits_complexity", repr(e))

print("Guard tests completed")

Guard tests completed


In [3]:
# logger.py tests
try:
    trace_id = f"nb-log-{uuid.uuid4()}"
    logger.log_event({"trace_id": trace_id, "event_type": "tool_ok", "tool_name": "notebook_logger_test", "sql_preview": "SELECT * FROM dbo.Users"})

    log_path = Path(os.getenv("LOG_PATH", "logs/events.jsonl"))
    if not log_path.exists():
        mark_fail("logger_writes_jsonl", f"Missing log file: {log_path}")
    else:
        last_line = log_path.read_text(encoding="utf-8").strip().splitlines()[-1]
        rec = json.loads(last_line)
        if rec.get("trace_id") == trace_id and rec.get("event_type") == "tool_ok":
            mark_ok("logger_writes_jsonl", f"line={last_line[:120]}")
        else:
            mark_fail("logger_writes_jsonl", f"Unexpected log record: {rec}")
except Exception as e:
    mark_fail("logger_writes_jsonl", repr(e))

print("Logger test completed")

Logger test completed


In [4]:
# DB + tools tests (depend on reachable SQL Server)
db_trace = f"nb-db-{uuid.uuid4()}"

try:
    with db_connection.get_connection() as conn:
        row = conn.execute(text("SELECT 1 AS ok")).first()
    if row and int(row[0]) == 1:
        mark_ok("db_connection", "SELECT 1 succeeded")
    else:
        mark_fail("db_connection", f"Unexpected row: {row}")
except Exception as e:
    mark_skip("db_connection", f"DB not reachable/configured: {e}")

if any(r["test"] == "db_connection" and r["status"] == "PASS" for r in RESULTS):
    try:
        schema = tools.get_schema(db_trace)
        mark_ok("tool_get_schema", f"tables={len(schema.get('tables', {}))}, views={len(schema.get('views', {}))}, rel={len(schema.get('relationships', []))}")
    except Exception as e:
        mark_fail("tool_get_schema", repr(e))

    try:
        q_res = tools.execute_readonly_sql(db_trace, "SELECT TOP (5) name FROM sys.tables")
        mark_ok("tool_execute_readonly_sql", f"rows={q_res.get('row_count')}")
    except Exception as e:
        mark_fail("tool_execute_readonly_sql", repr(e))

    try:
        first_table = next(iter(schema.get("tables", {})), None)
        if not first_table:
            raise ValueError("No base tables found for preview test")
        schema_name, table_name = first_table.split(".", 1)
        p_res = tools.preview_table(db_trace, table_name=table_name, schema_name=schema_name)
        mark_ok("tool_preview_table", f"table={schema_name}.{table_name}, rows={p_res.get('row_count')}")
    except Exception as e:
        mark_skip("tool_preview_table", f"Preview unavailable: {e}")
else:
    mark_skip("tool_get_schema", "Skipped because DB connection failed")
    mark_skip("tool_execute_readonly_sql", "Skipped because DB connection failed")
    mark_skip("tool_preview_table", "Skipped because DB connection failed")

# explain_reasoning is local-only and should always work
try:
    er = tools.explain_reasoning(db_trace, "List tables", ["sys.tables"], "SELECT TOP (5) name FROM sys.tables")
    mark_ok("tool_explain_reasoning", er.get("limit_policy", ""))
except Exception as e:
    mark_fail("tool_explain_reasoning", repr(e))

print("DB/tools tests completed")

DB/tools tests completed


In [5]:
# FastAPI route tests (in-process, no external server needed)
try:
    client = TestClient(mcp_server.app)

    h = client.get("/health")
    if h.status_code == 200:
        mark_ok("api_health", h.text)
    else:
        mark_fail("api_health", f"status={h.status_code}, body={h.text}")

    headers = {}
    key = os.getenv("MCP_API_KEY", "").strip()
    if key:
        headers["x-api-key"] = key

    er_body = {"question": "q", "chosen_tables": ["dbo.t"], "sql": "SELECT 1"}
    er_resp = client.post("/tools/explain_reasoning", headers=headers, json=er_body)
    if er_resp.status_code == 200:
        mark_ok("api_explain_reasoning", er_resp.text[:120])
    else:
        mark_fail("api_explain_reasoning", f"status={er_resp.status_code}, body={er_resp.text}")

    list_payload = {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
    list_resp = client.post("/mcp", headers=headers, json=list_payload)
    if list_resp.status_code == 200 and "result" in list_resp.json():
        mark_ok("api_mcp_tools_list", "tools/list returned result")
    else:
        mark_fail("api_mcp_tools_list", f"status={list_resp.status_code}, body={list_resp.text}")

except Exception as e:
    mark_fail("api_tests", repr(e))

print("API tests completed")

API tests completed


## Direct Tool Calls (One Tool Per Cell)

Run these cells to see each tool response directly.

In [6]:
direct_trace = f"nb-direct-{uuid.uuid4()}"
schema_result = tools.get_schema(direct_trace)
print("tables:", len(schema_result.get("tables", {})))
print("views:", len(schema_result.get("views", {})))
print("relationships:", len(schema_result.get("relationships", [])))
list(schema_result.get("tables", {}).keys())[:5]

tables: 6
views: 1
relationships: 0


['dbo.Dim_Customers',
 'dbo.Dim_Date',
 'dbo.Dim_Employees',
 'dbo.Dim_Orders',
 'dbo.Dim_Products']

In [7]:
direct_trace = f"nb-direct-{uuid.uuid4()}"
sql_result = tools.execute_readonly_sql(direct_trace, "SELECT TOP (5) name FROM sys.tables")
print("sanitized sql:", sql_result.get("sql"))
print("row_count:", sql_result.get("row_count"))
sql_result.get("rows", [])

sanitized sql: SELECT TOP (5) name FROM sys.tables
row_count: 5


[{'name': 'Dim_Products'},
 {'name': 'Dim_Employees'},
 {'name': 'Dim_Customers'},
 {'name': 'Dim_Orders'},
 {'name': 'Fact_Sales'}]

In [8]:
direct_trace = f"nb-direct-{uuid.uuid4()}"
schema_for_preview = tools.get_schema(direct_trace)
first_table = next(iter(schema_for_preview.get("tables", {})), None)
if not first_table:
    raise ValueError("No base table found for preview")
schema_name, table_name = first_table.split(".", 1)
preview_result = tools.preview_table(direct_trace, table_name=table_name, schema_name=schema_name)
print("table:", preview_result.get("table"))
print("row_count:", preview_result.get("row_count"))
preview_result.get("rows", [])

table: dbo.Dim_Customers
row_count: 5


[{'CustomerSK': 100,
  'CustomerBK': 'ALFKI',
  'CustomerName': 'Alfreds Futterkiste',
  'City': 'Berlin',
  'Region': 'Unknown',
  'Country': 'Germany'},
 {'CustomerSK': 101,
  'CustomerBK': 'ANATR',
  'CustomerName': 'Ana Trujillo Emparedados y helados',
  'City': 'México D.F.',
  'Region': 'Unknown',
  'Country': 'Mexico'},
 {'CustomerSK': 102,
  'CustomerBK': 'ANTON',
  'CustomerName': 'Antonio Moreno Taquería',
  'City': 'México D.F.',
  'Region': 'Unknown',
  'Country': 'Mexico'},
 {'CustomerSK': 103,
  'CustomerBK': 'AROUT',
  'CustomerName': 'Around the Horn',
  'City': 'London',
  'Region': 'Unknown',
  'Country': 'UK'},
 {'CustomerSK': 104,
  'CustomerBK': 'BERGS',
  'CustomerName': 'Berglunds snabbköp',
  'City': 'Luleå',
  'Region': 'Unknown',
  'Country': 'Sweden'}]

In [9]:
direct_trace = f"nb-direct-{uuid.uuid4()}"
explain_result = tools.explain_reasoning(
    direct_trace,
    question="Show me table names",
    chosen_tables=["sys.tables"],
    sql="SELECT TOP (5) name FROM sys.tables"
)
explain_result

{'interpretation': 'Question interpreted as: Show me table names',
 'tables_used': ['sys.tables'],
 'join_logic': 'No JOINs; single-source query.',
 'filters': 'No explicit filters.',
 'aggregations': ['none'],
 'limit_policy': 'Result size is bounded with TOP per policy and max table complexity.',
 'agent_think_ms': 0}

In [10]:
from collections import Counter

counts = Counter(r['status'] for r in RESULTS)
print("Test Summary:")
print(dict(counts))

for row in RESULTS:
    print(f"- [{row['status']}] {row['test']}: {row['details']}")

RESULTS

Test Summary:
{'PASS': 11}
- [PASS] guard_adds_top: SELECT TOP (200) name FROM sys.tables
- [PASS] guard_limits_complexity: Query exceeds max table limit (3).
- [PASS] logger_writes_jsonl: line={"trace_id": "nb-log-6228220f-67d7-49ae-8d68-8901caeee9b3", "event_type": "tool_ok", "tool_name": "notebook_logger_test"
- [PASS] db_connection: SELECT 1 succeeded
- [PASS] tool_get_schema: tables=6, views=1, rel=0
- [PASS] tool_execute_readonly_sql: rows=5
- [PASS] tool_preview_table: table=dbo.Dim_Customers, rows=5
- [PASS] tool_explain_reasoning: Result size is bounded with TOP per policy and max table complexity.
- [PASS] api_health: {"status":"ok"}
- [PASS] api_explain_reasoning: {"interpretation":"Question interpreted as: q","tables_used":["dbo.t"],"join_logic":"No JOINs; single-source query.","fi
- [PASS] api_mcp_tools_list: tools/list returned result


[{'test': 'guard_adds_top',
  'status': 'PASS',
  'details': 'SELECT TOP (200) name FROM sys.tables'},
 {'test': 'guard_limits_complexity',
  'status': 'PASS',
  'details': 'Query exceeds max table limit (3).'},
 {'test': 'logger_writes_jsonl',
  'status': 'PASS',
  'details': 'line={"trace_id": "nb-log-6228220f-67d7-49ae-8d68-8901caeee9b3", "event_type": "tool_ok", "tool_name": "notebook_logger_test"'},
 {'test': 'db_connection', 'status': 'PASS', 'details': 'SELECT 1 succeeded'},
 {'test': 'tool_get_schema',
  'status': 'PASS',
  'details': 'tables=6, views=1, rel=0'},
 {'test': 'tool_execute_readonly_sql', 'status': 'PASS', 'details': 'rows=5'},
 {'test': 'tool_preview_table',
  'status': 'PASS',
  'details': 'table=dbo.Dim_Customers, rows=5'},
 {'test': 'tool_explain_reasoning',
  'status': 'PASS',
  'details': 'Result size is bounded with TOP per policy and max table complexity.'},
 {'test': 'api_health', 'status': 'PASS', 'details': '{"status":"ok"}'},
 {'test': 'api_explain_reas